In [1]:
import math
import os
from dataclasses import dataclass

In [2]:
!pip install -U transformers datasets accelerate torch

In [3]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [37]:
class Config:
    # --- data ---
    data_path: str = "/content/shakespeare_2.txt"   # path to your corpus (plain .txt)
    seed: int = 42
    validation_split: float = 0.15  # ✅ match your other models

    #model_name: str = "distilgpt2"  # or "gpt2"
    model_name: str = "gpt2"
    block_size: int = 256

    output_dir: str = "/content/ft_shakespeare_gpt2"
    num_train_epochs: int = 20
    learning_rate: float = 5e-5
    weight_decay: float = 0.01

    #per_device_train_batch_size: int = 8
    per_device_train_batch_size: int = 4
    per_device_eval_batch_size: int = 8
    #gradient_accumulation_steps: int = 2
    gradient_accumulation_steps: int = 4

    eval_steps: int = 200
    save_steps: int = 200
    logging_steps: int = 50

    fp16: bool = True

In [38]:
cfg = Config()
set_seed(cfg.seed)

if not os.path.exists(cfg.data_path):
  raise FileNotFoundError(
      f"Could not find {cfg.data_path}. Put your Shakespeare corpus there or change Config.data_path."
  )

Load text dataset (loads line-by-line)

In [39]:
raw = load_dataset("text", data_files={"data": cfg.data_path})
raw = raw["data"].train_test_split(test_size=cfg.validation_split, seed=cfg.seed)
train_ds, eval_ds = raw["train"], raw["test"]

Call Tokenizer + model

In [40]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)

# GPT-2 family has no pad token by default; set pad_token to eos_token for batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(cfg.model_name)
model.resize_token_embeddings(len(tokenizer))

Embedding(50257, 768)

Tokenize:

In [41]:
def tokenize_fn(examples):
  return tokenizer(examples["text"], return_special_tokens_mask=False)

tokenized_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_eval = eval_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/34000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

In [42]:
import pandas as pd

df = pd.DataFrame(tokenized_train.select(range(5)))
df

,input_ids,attention_mask
0,"[6104, 588, 281, 267, 6, 6422, 2053, 18744, 28...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
1,"[1870, 351, 11906, 629, 19942, 9859, 338, 83, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
2,"[10248, 2612, 11, 290, 11, 1312, 6, 4562, 11, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
3,"[6423, 481, 314, 3830, 262, 7351, 12, 20123, 4...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
4,"[34, 4261, 51, 1797, 25]","[1, 1, 1, 1, 1]"


In [43]:
# Show first 3 examples (compact)
for i in range(3):
    ids = tokenized_train[i]["input_ids"]
    print(f"\n--- example {i} ---")
    print("ids:", ids)
    print("len:", len(ids))
    print("decoded snippet:", tokenizer.decode(ids[:120]))


--- example 0 ---
ids: [6104, 588, 281, 267, 6, 6422, 2053, 18744, 287, 257, 11527, 11]
len: 12
decoded snippet: Even like an o'ergrown lion in a cave,

--- example 1 ---
ids: [1870, 351, 11906, 629, 19942, 9859, 338, 83, 18180, 422, 465, 2951, 11]
len: 13
decoded snippet: And with thy scorns drew'st rivers from his eyes,

--- example 2 ---
ids: [10248, 2612, 11, 290, 11, 1312, 6, 4562, 11, 314, 481, 1560, 607, 355, 881, 25]
len: 16
decoded snippet: Good heart, and, i' faith, I will tell her as much:


Group tokens into fixed-size blocks for causal LM

In [44]:
def group_texts(examples):
  # Concatenate all token lists, then split into blocks
  concatenated = {k: sum(examples[k], []) for k in examples.keys()}
  total_len = len(concatenated["input_ids"])
  total_len = (total_len // cfg.block_size) * cfg.block_size  # drop remainder
  result = {
      k: [t[i : i + cfg.block_size] for i in range(0, total_len, cfg.block_size)]
      for k, t in concatenated.items()
  }
  # Labels are the same as input_ids for causal LM
  result["labels"] = result["input_ids"].copy()
  return result

lm_train = tokenized_train.map(group_texts, batched=True)
lm_eval = tokenized_eval.map(group_texts, batched=True)

Map:   0%|          | 0/34000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

In [45]:
# Show first 3 examples
for i in range(3):
    ids = lm_train[i]["input_ids"]
    print(f"\n--- example {i} ---")
    print("ids:", ids)
    print("len:", len(ids))
    print("decoded snippet:", tokenizer.decode(ids[:120]))


--- example 0 ---
ids: [6104, 588, 281, 267, 6, 6422, 2053, 18744, 287, 257, 11527, 11, 1870, 351, 11906, 629, 19942, 9859, 338, 83, 18180, 422, 465, 2951, 11, 10248, 2612, 11, 290, 11, 1312, 6, 4562, 11, 314, 481, 1560, 607, 355, 881, 25, 6423, 481, 314, 3830, 262, 7351, 12, 20123, 495, 338, 31322, 319, 34, 4261, 51, 1797, 25, 47731, 49, 52, 3398, 9399, 25, 40, 373, 3888, 351, 282, 13, 1870, 287, 326, 41998, 779, 340, 284, 262, 1918, 13, 1722, 890, 355, 345, 393, 314, 1865, 339, 1276, 4656, 13, 1870, 345, 1165, 11, 46123, 290, 5575, 2064, 11, 2061, 389, 484, 326, 6129, 612, 30, 15567, 3698, 46, 25, 34, 516, 259, 286, 3423, 3841, 11, 355, 11906, 2728, 318, 826, 11, 464, 40894, 995, 2314, 757, 5368, 1870, 326, 465, 10846, 25722, 82, 379, 465, 4369, 25, 33, 7115, 514, 1111, 290, 3758, 262, 5822, 351, 502, 13, 3844, 7062, 517, 11, 290, 3478, 1661, 517, 14142, 11, 4863, 543, 3253, 4335, 11, 262, 3872, 286, 644, 356, 389, 13681, 47995, 338, 16599, 287, 1966, 10861, 1528, 25, 10915, 407, 32

Before group_texts, each row corresponds to one original line from input text file.

After group_texts, each row becomes a fixed-length chunk of block_size tokens with labels

Data collator (for causal LM, set mlm=False)

In [47]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Train -

In [48]:
from transformers.trainer_utils import IntervalStrategy

# Training arguments
args = TrainingArguments(
  output_dir=cfg.output_dir,
  overwrite_output_dir=True,

  num_train_epochs=cfg.num_train_epochs,
  learning_rate=cfg.learning_rate,
  weight_decay=cfg.weight_decay,

  per_device_train_batch_size=cfg.per_device_train_batch_size,
  per_device_eval_batch_size=cfg.per_device_eval_batch_size,
  gradient_accumulation_steps=cfg.gradient_accumulation_steps,

  eval_strategy="steps",
  eval_steps=cfg.eval_steps,

  save_strategy="steps",
  save_steps=cfg.save_steps,
  save_total_limit=2,

  logging_strategy="steps",
  logging_steps=cfg.logging_steps,

  fp16=cfg.fp16,

  report_to="none",
  load_best_model_at_end=True,
  metric_for_best_model="eval_loss",
  greater_is_better=False,
)

# Trainer
trainer = Trainer(
  model=model,
  args=args,
  train_dataset=lm_train,
  eval_dataset=lm_eval,
  tokenizer=tokenizer,
  data_collator=data_collator,
)

# Train
trainer.train()

/tmp/ipython-input-4078845477.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss,Validation Loss
200,4.230000,4.271970
400,4.027300,4.233334
600,3.883500,4.225002
800,3.782800,4.240472
1000,3.707600,4.254265
1200,3.672200,4.262771


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=1220, training_loss=3.9461489505455143, metrics={'train_runtime': 806.4512, 'train_samples_per_second': 24.106, 'train_steps_per_second': 1.513, 'total_flos': 2539758551040000.0, 'train_loss': 3.9461489505455143, 'epoch': 20.0})

Evaluate & save

In [32]:
# Evaluate (perplexity) for distilgpt2
eval_out = trainer.evaluate()
eval_loss = eval_out.get("eval_loss")
ppl = math.exp(eval_loss) if eval_loss is not None else None
print(f"\nEval loss: {eval_loss:.4f}" if eval_loss is not None else "\nEval loss: n/a")
print(f"Perplexity: {ppl:.2f}" if ppl is not None else "Perplexity: n/a")

# Save
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

# Generate a few samples
prompt = "To be, or not to be,"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

model.eval()
with torch.no_grad():
  out = model.generate(
      **inputs,
      max_new_tokens=120,
      do_sample=True,
      temperature=0.9,
      top_p=0.95,
      pad_token_id=tokenizer.eos_token_id,
  )

print("\n--- SAMPLE ---")
print(tokenizer.decode(out[0], skip_special_tokens=True))


Eval loss: 4.3648
Perplexity: 78.64

--- SAMPLE ---
To be, or not to be, that I may not be?Thou art a prophet, I think, who lives, so often have I heard them speak.And now, sir, what a tale!What, now, what a tale! O, I am a villain!BAPTISTA:For, for, as I have heard, he shall be hanged.So he should live like a king,That hath made me no more content'd than thou, and thou, art:LADY GREY:Hath told me my father lived, but she had nothing.I cannot tell how much time nor


In [49]:
# Evaluate (perplexity) for gpt2
eval_out = trainer.evaluate()
eval_loss = eval_out.get("eval_loss")
ppl = math.exp(eval_loss) if eval_loss is not None else None
print(f"\nEval loss: {eval_loss:.4f}" if eval_loss is not None else "\nEval loss: n/a")
print(f"Perplexity: {ppl:.2f}" if ppl is not None else "Perplexity: n/a")

# Save
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

# Generate a few samples
prompt = "To be, or not to be,"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

model.eval()
with torch.no_grad():
  out = model.generate(
      **inputs,
      max_new_tokens=120,
      do_sample=True,
      temperature=0.9,
      top_p=0.95,
      pad_token_id=tokenizer.eos_token_id,
  )

print("\n--- SAMPLE ---")
print(tokenizer.decode(out[0], skip_special_tokens=True))


Eval loss: 4.2250
Perplexity: 68.37

--- SAMPLE ---
To be, or not to be, but to have:Hath she been wailing, or waking me asleep?What, is't Henry dead? or was that my father?MENENIUS:Thou dost mock me.To the duke's son and the king's wife, as you have done?But when the king's time comes, thou dost not wish me to stay.For how can I look upon the sun?I hear a gentleman that speaks it well, he makes me laugh.That is, as I can imagine, to be your good graces.BENVOLIO:FRI
